# Processamento de Sinais I — Aula Prática 3
## Questão 7 — Aproximações FIR e relação entre ordem e qualidade

Objetivo: estudar a relação entre ordem do filtro e qualidade da recuperação, tomando como base os filtros recuperadores das questões 3 e 6.

## Importar bibliotecas e helpers

In [3]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.io import wavfile
from scipy import signal


def load_calculate_spectrum():
    notebook_path = Path('../tools/calculate_spectrum.ipynb')
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))

    namespace = {}
    for cell in notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue

        source = ''.join(cell.get('source', []))
        exec(source, namespace)
        if 'calculate_spectrum' in namespace:
            return namespace['calculate_spectrum']

    raise RuntimeError('calculate_spectrum() nao encontrada em ../tools/calculate_spectrum.ipynb')


calculate_spectrum = load_calculate_spectrum()
plt.style.use('seaborn-v0_8-whitegrid')



def to_float_mono(data):
    data = np.asarray(data)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if np.issubdtype(data.dtype, np.integer):
        data = data.astype(np.float64) / np.iinfo(data.dtype).max
    else:
        data = data.astype(np.float64)
    peak = np.max(np.abs(data))
    if peak > 1:
        data = data / peak
    return data



def normalize_audio(audio):
    peak = np.max(np.abs(audio))
    if peak == 0:
        return audio
    return audio / peak



def show_audio(audio, rate, label):
    print(label)
    display(Audio(normalize_audio(audio), rate=rate))



def plot_frequency_response(b, a, title):
    w, h = signal.freqz(b, a, worN=2048)
    plt.figure(figsize=(12, 4))
    plt.plot(w / np.pi, 20 * np.log10(np.maximum(np.abs(h), 1e-8)))
    plt.title(title)
    plt.xlabel('Frequencia normalizada (x pi rad/amostra)')
    plt.ylabel('Magnitude (dB)')
    plt.tight_layout()
    return w, h



def plot_zplane(b, a, title):
    zeros, poles, gain = signal.tf2zpk(b, a)
    theta = np.linspace(0, 2 * np.pi, 512)

    plt.figure(figsize=(6, 6))
    plt.plot(np.cos(theta), np.sin(theta), '--', color='gray', label='Circulo unitario')
    if len(zeros):
        plt.scatter(np.real(zeros), np.imag(zeros), marker='o', facecolors='none', edgecolors='C0', s=100, label='Zeros')
    if len(poles):
        plt.scatter(np.real(poles), np.imag(poles), marker='x', color='C3', s=100, label='Polos')
    plt.axhline(0, color='black', linewidth=0.8)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title(title)
    plt.xlabel('Parte real')
    plt.ylabel('Parte imaginaria')
    plt.legend()
    plt.tight_layout()
    return zeros, poles, gain



def design_inverse_fir(b, a, num_taps=129, epsilon=1e-3, n_fft=8192):
    _, H = signal.freqz(b, a, worN=n_fft, whole=True)
    G = np.conj(H) / (np.abs(H) ** 2 + epsilon)
    h = np.fft.ifft(G).real
    h = h[:num_taps]
    h *= np.hamming(num_taps)
    return normalize_audio(h)



def align_signals(reference, estimate):
    corr = signal.correlate(estimate, reference, mode='full')
    lag = np.argmax(corr) - (len(reference) - 1)
    if lag > 0:
        estimate_aligned = estimate[lag:lag + len(reference)]
        reference_aligned = reference[:len(estimate_aligned)]
    else:
        start = -lag
        reference_aligned = reference[start:start + len(estimate)]
        estimate_aligned = estimate[:len(reference_aligned)]
    size = min(len(reference_aligned), len(estimate_aligned))
    return reference_aligned[:size], estimate_aligned[:size], lag



def quality_metrics(reference, estimate):
    reference_aligned, estimate_aligned, lag = align_signals(reference, estimate)
    error = reference_aligned - estimate_aligned
    mse = np.mean(error ** 2)
    snr = 10 * np.log10(np.mean(reference_aligned ** 2) / np.maximum(np.mean(error ** 2), 1e-12))
    corr = np.corrcoef(reference_aligned, estimate_aligned)[0, 1]
    return {'lag': lag, 'mse': mse, 'snr_db': snr, 'corr': corr}


## Variar a ordem dos filtros FIR e medir a qualidade da recuperação

In [4]:
fs, audio_int = wavfile.read('../data/handel.wav')
audio = to_float_mono(audio_int)
ordens = [16, 32, 64, 128]

sistemas = {
    'Q3': (np.array([1, 0, 0.49, 0, 0, 0, 0.2401, 0, -0.0576, 0, -0.0282, 0, -0.0138]), np.array([1.0])),
    'Q6_a0.7_L4': (np.array([1, 0, 0, 0, -1.0]), np.array([1, 0, 0, 0, -0.7])),
    'Q6_a0.9_L10': (np.r_[1.0, np.zeros(9), -1.0], np.r_[1.0, np.zeros(9), -0.9]),
}

for nome, (b, a) in sistemas.items():
    saida = signal.lfilter(b, a, audio)
    print(f'Sistema: {nome}')
    for ordem in ordens:
        g = design_inverse_fir(b, a, num_taps=ordem, epsilon=1e-3)
        recuperado = signal.lfilter(g, [1.0], saida)
        metrics = quality_metrics(audio, recuperado)
        print(f'  Ordem {ordem}:', metrics)


Sistema: Q3
  Ordem 16: {'lag': np.int64(9), 'mse': np.float64(0.035675240633283735), 'snr_db': np.float64(0.3321163102960125), 'corr': np.float64(0.3898677559947569)}
  Ordem 32: {'lag': np.int64(0), 'mse': np.float64(0.027740986474954935), 'snr_db': np.float64(1.4242854404178142), 'corr': np.float64(0.5300487502309709)}
  Ordem 64: {'lag': np.int64(0), 'mse': np.float64(0.005935891962803531), 'snr_db': np.float64(8.120644567577447), 'corr': np.float64(0.9252434695813224)}
  Ordem 128: {'lag': np.int64(0), 'mse': np.float64(0.00045218038343521565), 'snr_db': np.float64(19.30238726868765), 'corr': np.float64(0.9942075257136141)}
Sistema: Q6_a0.7_L4
  Ordem 16: {'lag': np.int64(8), 'mse': np.float64(0.018661943783288556), 'snr_db': np.float64(3.146189377455279), 'corr': np.float64(0.7219092552383201)}
  Ordem 32: {'lag': np.int64(0), 'mse': np.float64(0.0259127440081631), 'snr_db': np.float64(1.7203703998242392), 'corr': np.float64(0.5915362100211584)}
  Ordem 64: {'lag': np.int64(0), '

## Comentários

Em geral, ordens maiores permitem uma aproximação mais fiel da resposta inversa desejada, o que tende a melhorar métricas como correlação e SNR e a reduzir o erro médio quadrático.

Por outro lado, filtros maiores aumentam custo computacional e podem tornar a resposta mais sensível a erros de modelagem. Assim, há um compromisso prático entre complexidade e qualidade de recuperação.